<a href="https://colab.research.google.com/github/ojohnso3-oss/INST462/blob/main/Medium3_Code.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

df = pd.read_csv('/content/sample_data/anime.csv')
print(df.shape)   # (12294, 7)
print(df.columns.tolist())
# ['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members']

(12294, 7)
['anime_id', 'name', 'genre', 'type', 'episodes', 'rating', 'members']


In [10]:
from sklearn.preprocessing import MultiLabelBinarizer, MinMaxScaler
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import pandas as pd

# Drop rows with missing genre, rating, or type
df = df.dropna(subset=['genre', 'rating', 'type'])

# Clean episodes column
df['episodes_clean'] = pd.to_numeric(df['episodes'], errors='coerce').fillna(1)

# Parse genre lists
df['genre_list'] = df['genre'].apply(lambda x: [g.strip() for g in x.split(',')])

# One-hot encode genres (weight x2)
mlb = MultiLabelBinarizer()
genre_matrix = mlb.fit_transform(df['genre_list'])
genre_df = pd.DataFrame(genre_matrix, columns=mlb.classes_, index=df.index) * 2

# Type dummies
type_dummies = pd.get_dummies(df['type'], prefix='type')

# Scaled numerics
scaler = MinMaxScaler()
numeric = scaler.fit_transform(df[['rating', 'members', 'episodes_clean']])
numeric_df = pd.DataFrame(numeric,
    columns=['rating_scaled', 'members_scaled', 'episodes_scaled'],
    index=df.index)

# Combined feature matrix
feature_matrix = pd.concat([genre_df, type_dummies, numeric_df], axis=1)

# Verify no NaNs snuck through
assert feature_matrix.isnull().sum().sum() == 0, "NaNs still present!"

# Pairwise cosine similarity
sim_matrix = cosine_similarity(feature_matrix)
print(f"Feature matrix: {feature_matrix.shape}")
print(f"Similarity matrix: {sim_matrix.shape}")

Feature matrix: (12017, 52)
Similarity matrix: (12017, 12017)
